# Matimo Notebook 03: Meta-Tools — Agents That Create Their Own Tools

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tallclub/matimo/blob/main/docs/notebooks/03_meta_tools.ipynb)

This is the most advanced notebook in the series. You will watch a Matimo agent **define and register its own tools at runtime** — without any human intervention.

### What you'll learn
- What meta-tools are and why they matter
- How Matimo's `draft_tool` command works
- How the policy engine governs tool creation (only approved roles can draft tools)
- End-to-end demo: agent writes a YAML tool definition, saves it, and calls it

### Prerequisites
- Complete Notebook 01 (Quickstart) first
- An API key for any supported provider (see Step 2 below)

In [2]:
# Step 1: Install Matimo and dependencies
!pip install -q matimo langchain
print('Matimo installed!')


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Matimo installed!


## Step 2: Choose Your AI Provider & Set API Key

Matimo works with multiple providers. Pick the one you have access to:

| Provider | Free Tier? | Where to get a key |
|----------|------------|-------------------|
| **Gemini** ⭐ Recommended | ✓ Free (1,500 req/day, no credit card) | https://aistudio.google.com/ |
| OpenAI | Paid | https://platform.openai.com/api-keys |
| Anthropic | Paid | https://console.anthropic.com/ |

> **New to this?** Use Gemini — it's completely free and takes 60 seconds to set up.

In [6]:
import os
from getpass import getpass

# Choose your provider: 'gemini' (free), 'openai', or 'anthropic'
provider = input('Provider (gemini / openai / anthropic) [default: gemini]: ').strip().lower() or 'gemini'

KEY_MAP = {
 'gemini': ('GOOGLE_API_KEY', 'https://aistudio.google.com/'),
 'openai': ('OPENAI_API_KEY', 'https://platform.openai.com/api-keys'),
 'anthropic': ('ANTHROPIC_API_KEY', 'https://console.anthropic.com/'),
}

if provider not in KEY_MAP:
 raise ValueError(f'Unknown provider: {provider}. Choose gemini, openai, or anthropic.')

env_var, key_url = KEY_MAP[provider]
api_key = getpass(f'Paste your {provider} API key (get it at {key_url}): ')
os.environ[env_var] = api_key

print(f'✓ {provider} key set. Ready to go!')

✓ openai key set. Ready to go!


## Understanding Meta-Tools

In traditional AI frameworks, tools are **static** — a developer writes them, deploys them, and the agent uses them. Matimo breaks this pattern.

With Matimo's meta-tools, an agent can:
1. Identify a capability gap during a task
2. Draft a new YAML tool definition to fill that gap
3. Submit it for approval (or auto-approve in dev mode)
4. Register and use the new tool in the same session

> **This is only possible because Matimo tools are YAML-first.** The agent generates YAML (not code), which is safe, inspectable, and policy-governed.

In [4]:
# Step 3: Create a PolicyConfig that blocks command and function tools
#         These require HITL approval to execute
import tempfile, os
from matimo import PolicyConfig

# Create a temp directory for user-created tools
d = tempfile.mkdtemp()
print(f'✓ Temp directory for tools: {d}')

# Create a policy that requires approval for function execution
# (matimo_create_tool itself is already gated by being high-risk)
policy_config = PolicyConfig(
    allowed_domains=['jsonplaceholder.typicode.com', 'api.github.com'],
    allowed_http_methods=['GET', 'POST'],
    allow_command_tools=False,      # ← BLOCKS shell commands (creates creation-time gate)
    allow_function_tools=False,     # ← BLOCKS function execution (requires HITL approval)
    protected_namespaces=['matimo_'],
)

print('✓ Policy configuration:')
print(f'  • allow_command_tools: {policy_config.allow_command_tools}')
print(f'  • allow_function_tools: {policy_config.allow_function_tools}')
print(f'  • allowed_domains: {policy_config.allowed_domains}')

✓ Temp directory for tools: /var/folders/1d/5sj004_10236f7xwyyjy5z5w0000gn/T/tmp1ljf3l90
✓ Policy configuration:
  • allow_command_tools: False
  • allow_function_tools: False
  • allowed_domains: ['jsonplaceholder.typicode.com', 'api.github.com']


In [7]:
# Step 4: Initialize Matimo + Set HITL Approval Callback
from matimo import Matimo, get_global_approval_handler, set_global_matimo_instance

# Define async approval handler for HITL
async def auto_approve_handler(request):
    """Auto-approve tool creation for demo (in prod, this would be manual)"""
    tool_name = request.get('tool_name', 'unknown')
    print(f'\n🔒 [HITL APPROVAL] Requires human approval:')
    print(f'   Tool: {tool_name}')
    print(f'   ✓ Auto-approved for demo\n')
    return True  # True = approved, False = denied

# Initialize Matimo with policy config and mark tools dir as untrusted (user-created)
m = await Matimo.init(
    tool_paths=[d],
    auto_discover=True,
    policy_config=policy_config,
    untrusted_paths=[d],  # Mark user-created tools dir as untrusted
    log_level='silent'
)
set_global_matimo_instance(m)

# Install the HITL approval callback on the global handler
approval_handler = get_global_approval_handler()
approval_handler.set_approval_callback(auto_approve_handler)
print('✓ HITL approval handler installed')

tools = m.list_tools()
print(f'✓ Loaded {len(tools)} tools using {provider}')
print(f'✓ Meta-tools available: {[t.name for t in tools if t.name.startswith("matimo_")][:5]}...')

✓ HITL approval handler installed
✓ Loaded 18 tools using openai
✓ Meta-tools available: ['matimo_approve_tool', 'matimo_validate_skill', 'matimo_list_user_tools', 'matimo_create_tool', 'matimo_get_tool_status']...


## Live Demo: Agent Creates a Tool

We will ask the agent to create a simple calculator tool. Watch how it:
1. Realises no calculator tool exists
2. Calls `draft_tool` to write a YAML definition
3. Registers the tool automatically
4. Uses the new tool to answer the original question

In [8]:
# Step 5: Agent Creates the Tool
# Agent only sees matimo_create_tool — cannot wander, cannot loop
import time
from matimo import convert_tools_to_langchain
from langchain.agents import create_agent   # LangChain 1.x API

# Initialize LLM (shared across steps 5, 6, 7)
print(f'⏳ Initializing {provider} LLM...')
start = time.time()

if provider == 'gemini':
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0, timeout=60)
elif provider == 'openai':
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, timeout=60)
elif provider == 'anthropic':
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model='claude-3-sonnet-20240229', temperature=0, timeout=60)
else:
    raise ValueError(f'Unknown provider: {provider}')

print(f'✓ LLM ready ({time.time()-start:.1f}s)')

# Convert all tools (lc_tools_all reused and refreshed across steps 5, 6, 7)
lc_tools_all = convert_tools_to_langchain(m.list_tools(), m)

# This agent ONLY sees matimo_create_tool — cannot wander or loop on other tools
create_tools = [t for t in lc_tools_all if t.name == 'matimo_create_tool']
print(f'✓ Agent tools: {[t.name for t in create_tools]}')

# Pre-built YAML with correct indentation — passed verbatim so LLM never reformats it
TOOL_YAML = """name: user_info_fetcher
description: Fetch user information from a public API
version: '1.0.0'
parameters:
  user_id:
    type: string
    required: true
    description: User ID to fetch (1-10)
execution:
  type: http
  method: GET
  url: 'https://jsonplaceholder.typicode.com/users/{user_id}'
"""

# LangChain 1.x create_agent: system_prompt is a plain string, no ChatPromptTemplate needed
agent_create = create_agent(
    model=llm,
    tools=create_tools,
    system_prompt=(
        'You are a tool-creation agent. Your only available tool is matimo_create_tool. '
        'Call it exactly once using the name, target_dir and yaml_content given by the user. '
        'Do not modify the yaml_content. Stop immediately after the call.'
    ),
)

print('\n' + '='*70)
print('PHASE 1: AGENT CREATES THE TOOL  (policy gate → HITL approval → write YAML)')
print('='*70)
print(f'Agent has: [matimo_create_tool] only  |  target_dir="{d}"  |  recursion_limit=5')
print('Policy: matimo_create_tool requires_approval=true → 🔒 HITL fires before execution\n')

invoke_start = time.time()
try:
    result_create = await agent_create.ainvoke({
        'messages': [{
            'role': 'user',
            'content': (
                f'Call matimo_create_tool with EXACTLY these arguments:\n'
                f'  name = "user_info_fetcher"\n'
                f'  target_dir = "{d}"\n'
                f'  yaml_content = the YAML below (copy verbatim, do not change indentation)\n\n'
                f'{TOOL_YAML}'
            )
        }]
    }, config={'recursion_limit': 5})
    print(f'✓ Agent finished in {time.time()-invoke_start:.1f}s')
except Exception as e:
    print(f'⚠ Agent error: {type(e).__name__}: {str(e)[:200]}')
    result_create = None

# Verify
if result_create:
    tools_called = [
        call['name']
        for msg in result_create.get('messages', [])
        if hasattr(msg, 'tool_calls') and msg.tool_calls
        for call in msg.tool_calls
    ]
    if 'matimo_create_tool' in tools_called:
        print('\n✅ matimo_create_tool was called')
        print(f'   Calls: {tools_called}')
        final = result_create['messages'][-1]
        if hasattr(final, 'content'):
            print(f'   Agent: {final.content[:200]}')
    else:
        print(f'\n⚠️  matimo_create_tool was NOT called. Calls: {tools_called}')
else:
    print('\n❌ Agent did not complete')

⏳ Initializing openai LLM...
✓ LLM ready (0.3s)
✓ Agent tools: ['matimo_create_tool']

PHASE 1: AGENT CREATES THE TOOL  (policy gate → HITL approval → write YAML)
Agent has: [matimo_create_tool] only  |  target_dir="/var/folders/1d/5sj004_10236f7xwyyjy5z5w0000gn/T/tmp1ljf3l90"  |  recursion_limit=5
Policy: matimo_create_tool requires_approval=true → 🔒 HITL fires before execution

✓ Agent finished in 6.2s

✅ matimo_create_tool was called
   Calls: ['matimo_create_tool']
   Agent: The tool "user_info_fetcher" has been successfully created and auto-approved. You can find it at the following path: 

`/var/folders/1d/5sj004_10236f7xwyyjy5z5w0000gn/T/tmp1ljf3l90/user_info_fetcher/d


In [21]:
# Step 6: Agent Hot-Reloads the Registry
# Agent only sees matimo_reload_tools + matimo_list_user_tools
import os, glob
print('\n' + '='*70)
print('PHASE 2: AGENT HOT-RELOADS THE REGISTRY')
print('='*70)

reload_tools = [t for t in lc_tools_all if t.name in ('matimo_reload_tools', 'matimo_list_user_tools')]
print(f'Agent tools: {[t.name for t in reload_tools]}  |  recursion_limit=5\n')

# LangChain 1.x create_agent: system_prompt is a plain string
agent_reload = create_agent(
    model=llm,
    tools=reload_tools,
    system_prompt=(
        'You are a registry-reload agent. '
        'Your tools are matimo_reload_tools and matimo_list_user_tools. '
        f'Step 1: call matimo_reload_tools with no arguments. '
        f'Step 2: call matimo_list_user_tools with tool_dir="{d}" and report what user tools exist. '
        'If the result contains a "failures" key, report those errors. '
        'Stop after reporting.'
    ),
)

invoke_start = time.time()
try:
    result_reload = await agent_reload.ainvoke({
        'messages': [{
            'role': 'user',
            'content': (
                f'Reload the tool registry, then list all user-created tools in tool_dir="{d}". '
                f'Pass tool_dir="{d}" explicitly when calling matimo_list_user_tools.'
            )
        }]
    }, config={'recursion_limit': 5})
    print(f'✓ Agent finished in {time.time()-invoke_start:.1f}s')
except Exception as e:
    print(f'⚠ Agent error: {type(e).__name__}: {str(e)[:150]}')
    result_reload = None

if result_reload:
    tools_called = [
        call['name']
        for msg in result_reload.get('messages', [])
        if hasattr(msg, 'tool_calls') and msg.tool_calls
        for call in msg.tool_calls
    ]
    print(f'Calls made: {tools_called}')
    print('✅ matimo_reload_tools called' if 'matimo_reload_tools' in tools_called else '⚠️  matimo_reload_tools NOT called')
    print('✅ matimo_list_user_tools called' if 'matimo_list_user_tools' in tools_called else '⚠️  matimo_list_user_tools NOT called')
    final = result_reload['messages'][-1]
    if hasattr(final, 'content'):
        print(f'\nAgent: {final.content[:300]}')
else:
    print('❌ Agent did not complete')

# ── Direct filesystem check (source of truth — bypasses agent/listing bugs) ──
print('\n─── Filesystem check (direct Python) ───')
yaml_files = glob.glob(os.path.join(d, '*', 'definition.yaml'))
if yaml_files:
    for f in yaml_files:
        tool_name_on_disk = os.path.basename(os.path.dirname(f))
        print(f'  ✅ Found on disk: {tool_name_on_disk}/definition.yaml')
        # Read and show first line to confirm it's valid YAML
        try:
            import yaml as _yaml
            with open(f) as fh:
                data = _yaml.safe_load(fh)
            print(f'     name={data.get("name")}  status={data.get("status")}  url={data.get("execution", {}).get("url", "")[:50]}')
        except Exception as e:
            print(f'     ⚠ parse error: {e}')
else:
    print(f'  ❌ No definition.yaml files found under {d}/')

# Rebuild lc_tools_all AFTER reload so user_info_fetcher is available in Step 7
await m.reload()
lc_tools_all = convert_tools_to_langchain(m.list_tools(), m)
fetcher_present = any(t.name == 'user_info_fetcher' for t in lc_tools_all)
print(f'\n✓ lc_tools_all rebuilt — user_info_fetcher present: {fetcher_present}')


PHASE 2: AGENT HOT-RELOADS THE REGISTRY
Agent tools: ['matimo_list_user_tools', 'matimo_reload_tools']  |  recursion_limit=5

✓ Agent finished in 4.1s
Calls made: ['matimo_reload_tools', 'matimo_list_user_tools']
✅ matimo_reload_tools called
✅ matimo_list_user_tools called

Agent: The tool registry has been successfully reloaded, with 1 tool loaded and 18 tools revalidated. However, there are no user-created tools found in the specified directory (tool_dir="/var/folders/1d/5sj004_10236f7xwyyjy5z5w0000gn/T/tmpb6wmrkb9").

✓ lc_tools_all rebuilt — user_info_fetcher present: True


In [22]:
# Step 7: Agent Executes the Created Tool
# Agent only sees user_info_fetcher — single-purpose, no drift
print('\n' + '='*70)
print('PHASE 3: AGENT EXECUTES THE TOOL')
print('='*70)

use_tools = [t for t in lc_tools_all if t.name == 'user_info_fetcher']

if not use_tools:
    print('❌ user_info_fetcher not found in lc_tools_all.')
    print('   Run Step 6 first so the registry is reloaded, then re-run this cell.')
else:
    print(f'Agent tools: {[t.name for t in use_tools]}  |  recursion_limit=5')
    print('Policy: execution of untrusted tools triggers 🔒 HITL approval\n')

    # LangChain 1.x create_agent: system_prompt is a plain string
    agent_use = create_agent(
        model=llm,
        tools=use_tools,
        system_prompt=(
            'You are a task-execution agent. Your only tool is user_info_fetcher. '
            'Call it with user_id="5". Report the name, email and city from the result. '
            'If the system requests approval, it is handled automatically. Stop after reporting.'
        ),
    )

    invoke_start = time.time()
    try:
        result_use = await agent_use.ainvoke({
            'messages': [{
                'role': 'user',
                'content': 'Call user_info_fetcher with user_id="5" and report the user information.'
            }]
        }, config={'recursion_limit': 5})
        print(f'✓ Agent finished in {time.time()-invoke_start:.1f}s')
    except Exception as e:
        print(f'⚠ Agent error: {type(e).__name__}: {str(e)[:150]}')
        result_use = None

    if result_use:
        tools_called = [
            call['name']
            for msg in result_use.get('messages', [])
            if hasattr(msg, 'tool_calls') and msg.tool_calls
            for call in msg.tool_calls
        ]
        if 'user_info_fetcher' in tools_called:
            print('\n✅ user_info_fetcher was called!')
            final = result_use['messages'][-1]
            if hasattr(final, 'content'):
                print(f'\n📝 Result:\n{final.content[:500]}')
        else:
            print(f'\n⚠️  user_info_fetcher was NOT called. Calls: {tools_called}')
    else:
        print('\n❌ Agent did not complete')


PHASE 3: AGENT EXECUTES THE TOOL
Agent tools: ['user_info_fetcher']  |  recursion_limit=5
Policy: execution of untrusted tools triggers 🔒 HITL approval

✓ Agent finished in 5.1s

✅ user_info_fetcher was called!

📝 Result:
The user information is as follows:

- Name: Chelsey Dietrich
- Email: Lucio_Hettinger@annie.ca
- City: Roscoeview


In [23]:
# Step 8: Verify All Three Phases Succeeded
import glob
import os

print('\n' + '='*70)
print('FINAL VERIFICATION: Three-Phase Workflow Complete?')
print('='*70)

# Check Phase 1: Tool file created on disk
print('\n📋 Phase 1 - Tool Creation (Check disk):')
all_yaml = glob.glob(os.path.join(d, '*/*.yaml'))  # Look in subdirs
tool_yamls = [f for f in all_yaml if 'policy' not in f.lower()]

if tool_yamls:
    print(f'✅ {len(tool_yamls)} tool file(s) found on disk:')
    for tool_file in tool_yamls:
        parent_dir = os.path.basename(os.path.dirname(tool_file))
        print(f'   • {parent_dir}/definition.yaml')
        try:
            with open(tool_file) as f:
                import re
                content = f.read()
                name_match = re.search(r'name:\s*(\w+)', content)
                if name_match:
                    print(f'     └─ Name: {name_match.group(1)}')
        except Exception:
            pass
else:
    print('❌ No tool files found on disk')

# Check Phase 2: Tool in registry
print('\n📋 Phase 2 - Registry Hot-Reload (Check registry):')
await m.reload()
tools_current = m.list_tools()
user_tools = [t for t in tools_current if t.name == 'user_info_fetcher']

if user_tools:
    print(f'✅ user_info_fetcher is in the registry')
    tool = user_tools[0]
    print(f'   • Description: {tool.description}')
else:
    print('❌ user_info_fetcher NOT found in registry')

# Check Phase 3: Tool can be executed
print('\n📋 Phase 3 - Tool Execution (Direct test call):')
try:
    result = await m.execute('user_info_fetcher', {'user_id': '5'})
    if isinstance(result, dict) and 'name' in result:
        print(f'✅ Tool executed successfully!')
        print(f'   • Name: {result.get("name")}')
        print(f'   • Email: {result.get("email")}')
        print(f'   • City: {result.get("address", {}).get("city")}')
    else:
        print(f'✅ Tool executed (response: {str(result)[:100]})')
except Exception as e:
    print(f'❌ Tool execution failed: {str(e)[:150]}')

# Final summary
print('\n' + '='*70)
print('🎯 WORKFLOW SUMMARY:')
print('='*70)

phase1_ok = len(tool_yamls) > 0
phase2_ok = len(user_tools) > 0
phase3_ok = False
try:
    result = await m.execute('user_info_fetcher', {'user_id': '1'})
    phase3_ok = isinstance(result, dict) and 'name' in result
except Exception:
    phase3_ok = False

print(f'\n  {"✅" if phase1_ok else "❌"} Phase 1: Tool YAML written to disk')
print(f'  {"✅" if phase2_ok else "❌"} Phase 2: Tool hot-reloaded into registry')
print(f'  {"✅" if phase3_ok else "❌"} Phase 3: Tool executed and returned data')

if phase1_ok and phase2_ok and phase3_ok:
    print('\n🎉 SUCCESS! Three-phase meta-tool workflow completed!')
    print('\n📚 What you just witnessed:')
    print('   1. AI agent called matimo_create_tool → 🔒 HITL gate fired → approved → YAML written')
    print('   2. AI agent called matimo_reload_tools → new tool registered in memory')
    print('   3. AI agent called user_info_fetcher → 🔒 HITL gate fired → approved → data returned')
    print('\n✨ This is Matimo policy-governed meta-tools in action!')
else:
    print('\n⚠️  Workflow incomplete. Check:')
    if not phase1_ok:
        print(f'   • Phase 1: matimo_create_tool should write YAML to {d}/<name>/definition.yaml')
    if not phase2_ok:
        print('   • Phase 2: matimo_reload_tools must be called after tool creation')
    if not phase3_ok:
        print('   • Phase 3: user_info_fetcher must be in registry and policy must allow execution')


FINAL VERIFICATION: Three-Phase Workflow Complete?

📋 Phase 1 - Tool Creation (Check disk):
✅ 1 tool file(s) found on disk:
   • user_info_fetcher/definition.yaml
     └─ Name: user_info_fetcher

📋 Phase 2 - Registry Hot-Reload (Check registry):
✅ user_info_fetcher is in the registry
   • Description: Fetch user information from a public API

📋 Phase 3 - Tool Execution (Direct test call):
✅ Tool executed successfully!
   • Name: Chelsey Dietrich
   • Email: Lucio_Hettinger@annie.ca
   • City: Roscoeview

🎯 WORKFLOW SUMMARY:

  ✅ Phase 1: Tool YAML written to disk
  ✅ Phase 2: Tool hot-reloaded into registry
  ✅ Phase 3: Tool executed and returned data

🎉 SUCCESS! Three-phase meta-tool workflow completed!

📚 What you just witnessed:
   1. AI agent called matimo_create_tool → 🔒 HITL gate fired → approved → YAML written
   2. AI agent called matimo_reload_tools → new tool registered in memory
   3. AI agent called user_info_fetcher → 🔒 HITL gate fired → approved → data returned

✨ This i

---
## Summary

You just saw Matimo's most powerful feature: **agents that create their own tools at runtime.**

| Feature | Status |
|---------|--------|
| Agent-authored tools | ✓ |
| Policy-governed tool creation | ✓ |
| YAML-only (no code execution) | ✓ |
| Blocked in prod environment | ✓ |
| Works with Gemini, OpenAI, Anthropic | ✓ |

### What's Next?
- Explore the full [Matimo docs](https://github.com/tallclub/matimo)
- Free Gemini key: https://aistudio.google.com/
- GitHub: https://github.com/tallclub/matimo